
## Folder 06 — Referential Integrity Validation


**Referential integrity** ensures that `records in one table reference records` that actually `exist in another table`.


For this e-commerce project, the main relationships are:

- **Orders → Customers**
- **Order Items → Orders**
- **Order Items → Products**
- **Payments → Orders**
- **Shipments → Orders**
- **Inventory → Products**

For example, if an order references CUST99999, but that customer does not exist in the customers table, that order has an orphan customer reference.

This notebook detects those problems and stores the results in **Delta tables**.


### Referential Integrity Architecture


```text
┌──────────────────────────────────────────────┐
│              SILVER DELTA TABLES             │
│                                              │
│  Customers · Products · Orders               │
│  Order Items · Payments · Shipments           │
│  Inventory                                   │
└──────────────────────────────────────────────┘
                       │
                       ▼
┌──────────────────────────────────────────────┐
│         REFERENTIAL INTEGRITY VALIDATION     │
│                                              │
│  • Validate Foreign Key References           │
│  • Match Source FK with Target Key           │
│  • Identify Missing Parent Records           │
└──────────────────────────────────────────────┘
                       │
              ┌────────┴────────┐
              ▼                 ▼
┌──────────────────────┐  ┌──────────────────────┐
│  RELATIONSHIP REPORT │  │   ORPHAN RECORDS     │
│                      │  │                      │
│  Validation Results  │  │  Missing References │
│  Orphan Counts       │  │  Source Record Data │
│  PASS / FAIL Status  │  │  Relationship Name  │
└──────────────────────┘  └──────────────────────┘
```

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "ecommerce_silver"

REPORT_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_report"
)

ORPHAN_TABLE = (
    f"{CATALOG}.{SCHEMA}.referential_integrity_orphans"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

print("Run ID:", RUN_ID)


## 3. Load Silver tables

TABLES = {
    "customers": f"{CATALOG}.{SCHEMA}.silver_customers",
    "products": f"{CATALOG}.{SCHEMA}.silver_products",
    "orders": f"{CATALOG}.{SCHEMA}.silver_orders",
    "order_items": f"{CATALOG}.{SCHEMA}.silver_order_items",
    "payments": f"{CATALOG}.{SCHEMA}.silver_payments",
    "shipments": f"{CATALOG}.{SCHEMA}.silver_shipments",
    "inventory": f"{CATALOG}.{SCHEMA}.silver_inventory"
}

for dataset_name, table_name in TABLES.items():

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Required Silver table does not exist: {table_name}"
        )

    print(f"Available: {table_name}")

customers_df = spark.table(TABLES["customers"])
products_df = spark.table(TABLES["products"])
orders_df = spark.table(TABLES["orders"])
order_items_df = spark.table(TABLES["order_items"])
payments_df = spark.table(TABLES["payments"])
shipments_df = spark.table(TABLES["shipments"])
inventory_df = spark.table(TABLES["inventory"])

print("All Silver tables loaded successfully.")


## 4. Define foreign-key relationships

RELATIONSHIPS = [

    {
        "relationship": "orders_to_customers",
        "source_dataset": "orders",
        "source_fk": "customer_id",
        "source_pk": "order_id",
        "target_dataset": "customers",
        "target_key": "customer_id"
    },

    {
        "relationship": "order_items_to_orders",
        "source_dataset": "order_items",
        "source_fk": "order_id",
        "source_pk": "order_item_id",
        "target_dataset": "orders",
        "target_key": "order_id"
    },

    {
        "relationship": "order_items_to_products",
        "source_dataset": "order_items",
        "source_fk": "product_id",
        "source_pk": "order_item_id",
        "target_dataset": "products",
        "target_key": "product_id"
    },

    {
        "relationship": "payments_to_orders",
        "source_dataset": "payments",
        "source_fk": "order_id",
        "source_pk": "payment_id",
        "target_dataset": "orders",
        "target_key": "order_id"
    },

    {
        "relationship": "shipments_to_orders",
        "source_dataset": "shipments",
        "source_fk": "order_id",
        "source_pk": "shipment_id",
        "target_dataset": "orders",
        "target_key": "order_id"
    },

    {
        "relationship": "inventory_to_products",
        "source_dataset": "inventory",
        "source_fk": "product_id",
        "source_pk": "inventory_snapshot_id",
        "target_dataset": "products",
        "target_key": "product_id"
    }
]


## 5. Define validation function

def validate_relationship(config):

    relationship = config["relationship"]

    source_dataset = config["source_dataset"]
    source_fk = config["source_fk"]
    source_pk = config["source_pk"]

    target_dataset = config["target_dataset"]
    target_key = config["target_key"]

    source_df = spark.table(TABLES[source_dataset])
    target_df = spark.table(TABLES[target_dataset])

    # Validate required columns.

    if source_fk not in source_df.columns:
        raise Exception(
            f"Missing source foreign key: {source_dataset}.{source_fk}"
        )

    if source_pk not in source_df.columns:
        raise Exception(
            f"Missing source record identifier: {source_dataset}.{source_pk}"
        )

    if target_key not in target_df.columns:
        raise Exception(
            f"Missing target key: {target_dataset}.{target_key}"
        )

    # Count source and target records.

    source_count = source_df.count()
    target_count = target_df.count()

    # Identify null foreign keys.

    null_fk_df = source_df.filter(
        F.col(source_fk).isNull()
        | (
            F.trim(
                F.col(source_fk).cast("string")
            ) == ""
        )
    )

    null_fk_count = null_fk_df.count()

    # Identify duplicate target keys.

    duplicate_target_keys_df = (
        target_df
        .groupBy(target_key)
        .count()
        .filter(
            F.col(target_key).isNotNull()
            & (F.col("count") > 1)
        )
    )

    duplicate_target_key_groups = (
        duplicate_target_keys_df.count()
    )

    # Build target reference lookup.

    target_keys_df = (
        target_df
        .select(
            F.col(target_key).alias("_reference_key")
        )
        .filter(
            F.col("_reference_key").isNotNull()
            & (
                F.trim(
                    F.col("_reference_key").cast("string")
                ) != ""
            )
        )
        .distinct()
    )

    # Exclude null source foreign keys from orphan detection.

    valid_fk_source_df = source_df.filter(
        F.col(source_fk).isNotNull()
        & (
            F.trim(
                F.col(source_fk).cast("string")
            ) != ""
        )
    )

    # Left anti join identifies source records without a parent.

    orphan_df = (
        valid_fk_source_df.alias("source")
        .join(
            target_keys_df.alias("target"),
            F.col(f"source.{source_fk}") ==
            F.col("target._reference_key"),
            "left_anti"
        )
    )

    orphan_count = orphan_df.count()

    # Store orphan records in a common JSON structure.

    source_columns = source_df.columns

    orphan_detail_df = (
        orphan_df
        .select(
            F.lit(relationship).alias("relationship"),

            F.lit(source_dataset).alias("source_dataset"),

            F.lit(source_fk).alias("foreign_key_column"),

            F.lit(target_dataset).alias("target_dataset"),

            F.lit(target_key).alias("target_key_column"),

            F.col(source_pk).cast("string").alias(
                "source_record_id"
            ),

            F.col(source_fk).cast("string").alias(
                "invalid_reference_value"
            ),

            F.to_json(
                F.struct(
                    *[
                        F.col(column_name)
                        for column_name in source_columns
                    ]
                )
            ).alias("source_record_json"),

            F.lit(RUN_ID).alias("run_id"),

            F.current_timestamp().alias("detected_at")
        )
    )

    # Record null foreign keys separately.

    null_fk_detail_df = (
        null_fk_df
        .select(
            F.lit(relationship).alias("relationship"),

            F.lit(source_dataset).alias("source_dataset"),

            F.lit(source_fk).alias("foreign_key_column"),

            F.lit(target_dataset).alias("target_dataset"),

            F.lit(target_key).alias("target_key_column"),

            F.col(source_pk).cast("string").alias(
                "source_record_id"
            ),

            F.col(source_fk).cast("string").alias(
                "invalid_reference_value"
            ),

            F.to_json(
                F.struct(
                    *[
                        F.col(column_name)
                        for column_name in source_columns
                    ]
                )
            ).alias("source_record_json"),

            F.lit(RUN_ID).alias("run_id"),

            F.current_timestamp().alias("detected_at")
        )
    )

    all_invalid_df = (
        orphan_detail_df
        .unionByName(null_fk_detail_df)
    )

    # Relationship status.

    status = (
        "PASS"
        if (
            null_fk_count == 0
            and orphan_count == 0
            and duplicate_target_key_groups == 0
        )
        else "FAIL"
    )

    result = {

        "relationship": relationship,

        "source_dataset": source_dataset,

        "source_foreign_key": source_fk,

        "target_dataset": target_dataset,

        "target_key": target_key,

        "source_records": source_count,

        "target_records": target_count,

        "null_foreign_key_records": null_fk_count,

        "orphan_records": orphan_count,

        "duplicate_target_key_groups": duplicate_target_key_groups,

        "status": status,

        "run_id": RUN_ID,

        "checked_at": datetime.now()
    }

    return result, all_invalid_df


## 6. Execute referential integrity checks

validation_results = []
orphan_dataframes = []

for config in RELATIONSHIPS:

    print(
        f"Checking relationship: {config['relationship']}"
    )

    result, invalid_df = validate_relationship(config)

    validation_results.append(result)

    orphan_dataframes.append(invalid_df)

    print("Source records:", result["source_records"])
    print("Null foreign keys:", result["null_foreign_key_records"])
    print("Orphan records:", result["orphan_records"])
    print("Duplicate target key groups:", result["duplicate_target_key_groups"])
    print("Status:", result["status"])

    print("-" * 60)


## 7. Create relationship validation report

report_df = spark.createDataFrame(
    validation_results
)

display(
    report_df.orderBy("relationship")
)


## 8. Combine all invalid reference records

combined_orphans_df = orphan_dataframes[0]

for next_df in orphan_dataframes[1:]:

    combined_orphans_df = (
        combined_orphans_df.unionByName(next_df)
    )

display(
    combined_orphans_df
)


## 9. Save referential integrity report

(
    report_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REPORT_TABLE)
)

print("Report saved:", REPORT_TABLE)


## 10. Save orphan records

# This table contains records with missing or invalid references.
# It represents the latest validation run.

(
    combined_orphans_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(ORPHAN_TABLE)
)

print("Orphan records saved:", ORPHAN_TABLE)


## 11. Relationship-level summary

summary_df = (
    report_df
    .agg(

        F.count("*").alias("total_relationships"),

        F.sum(
            F.when(
                F.col("status") == "PASS",
                1
            ).otherwise(0)
        ).alias("passed_relationships"),

        F.sum(
            F.when(
                F.col("status") == "FAIL",
                1
            ).otherwise(0)
        ).alias("failed_relationships"),

        F.sum("null_foreign_key_records").alias(
            "total_null_foreign_keys"
        ),

        F.sum("orphan_records").alias(
            "total_orphan_records"
        ),

        F.sum("duplicate_target_key_groups").alias(
            "total_duplicate_target_key_groups"
        )
    )
    .withColumn(
        "overall_status",
        F.when(
            F.col("failed_relationships") == 0,
            "PASS"
        ).otherwise("FAIL")
    )
)

display(summary_df)


## 12. Inspect orphan records

print("Orphan and null-reference records:")

display(
    spark.table(ORPHAN_TABLE)
    .orderBy(
        "relationship",
        "invalid_reference_value"
    )
)


## 13. Inspect duplicate parent keys

for config in RELATIONSHIPS:

    target_table = TABLES[config["target_dataset"]]
    target_key = config["target_key"]

    target_df = spark.table(target_table)

    duplicate_keys_df = (
        target_df
        .groupBy(target_key)
        .count()
        .filter(
            F.col(target_key).isNotNull()
            & (F.col("count") > 1)
        )
    )

    if duplicate_keys_df.limit(1).count() > 0:

        print(
            f"Duplicate keys in {target_table}.{target_key}"
        )

        display(duplicate_keys_df)


## 14. Final validation summary

total_relationships = report_df.count()

passed_relationships = (
    report_df
    .filter(F.col("status") == "PASS")
    .count()
)

failed_relationships = (
    report_df
    .filter(F.col("status") == "FAIL")
    .count()
)

total_orphans = (
    spark.table(ORPHAN_TABLE).count()
)

print("=" * 65)
print("REFERENTIAL INTEGRITY VALIDATION SUMMARY")
print("=" * 65)

print("Total relationships :", total_relationships)
print("Passed relationships:", passed_relationships)
print("Failed relationships:", failed_relationships)
print("Invalid reference records:", total_orphans)

print("=" * 65)

if failed_relationships > 0:

    print("WARNING: Referential integrity issues detected.")

else:

    print("All configured relationships passed.")

print("Report table:", REPORT_TABLE)
print("Orphan table:", ORPHAN_TABLE)


## 2. Expected Relationship Report


*Illustrative output — actual counts depend on your Silver Delta tables.*

| Relationship | Source Foreign Key (FK) | Target Primary/Business Key |
|---|---|---|
| Orders → Customers | `customer_id` | `customer_id` |
| Order Items → Orders | `order_id` | `order_id` |
| Order Items → Products | `product_id` | `product_id` |
| Payments → Orders | `order_id` | `order_id` |
| Shipments → Orders | `order_id` | `order_id` |
| Inventory → Products | `product_id` | `product_id` |


### Referential Integrity Report Tables


Two Delta tables are created to store validation results.

| Delta Table | Purpose |
|---|---|
| `referential_integrity_report` | Stores relationship-wise validation results, orphan counts, and validation status. |
| `referential_integrity_orphans` | Stores individual orphan records with missing parent references. |


### Why use a LEFT ANTI JOIN?

In [0]:
orphan_df = (
    source_df
    .join(
        target_df,
        source_df.customer_id == target_df.customer_id,
        "left_anti"
    )
)


A **left anti join** returns `source records` that have **no matching record** in the **target table**.


### Why check duplicate target keys?


- If the customers table contains two rows with the same customer_id, joining orders to customers could multiply order records. 
- A parent key should be unique for a conventional one-to-many relationship.